# Web Scraping & Product Price Analysis – Air Conditioners (Amazon India)
**Stage 8 – Results & Interpretation** (simplified, beginner version)

Pipeline position: **8 of 8**

## Objective
Turn the numbers into plain-English **findings** for buyers and sellers, and save them in a table.

## Step 1 – Setup

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Folder that holds all the CSV files of the project.
# (In Google Colab: upload the CSV files and change this to "/content/")
DATA = "../0. DataSet/"

print("Setup complete.")

Setup complete.


In [11]:
raw = pd.read_csv(  "Air_Conditioners_Raw.csv")
filtered = pd.read_csv( "Air_Conditioners_Filtered.csv")
df = pd.read_csv( "Air_Conditioners_Cleaned.csv")

print("Raw:", raw.shape, "| Filtered:", filtered.shape, "| Cleaned:", df.shape)

Raw: (32, 9) | Filtered: (32, 9) | Cleaned: (652, 15)


## Step 2 – How much data survived each stage?

In [12]:
funnel = pd.DataFrame({
    "Stage": ["Scraped (raw) listings", "After AC-only + unique filter", "After validation & cleaning",
              "... with a selling price", "... with verified MRP / discount", "... with a customer rating"],
    "Listings": [len(raw), len(filtered), len(df), df["discount_price"].notna().sum(),
                 df["discount_percent"].notna().sum(), df["ratings"].notna().sum()],
})
funnel

,Stage,Listings
0,Scraped (raw) listings,32
1,After AC-only + unique filter,32
2,After validation & cleaning,652
3,... with a selling price,423
4,... with verified MRP / discount,423
5,... with a customer rating,392


## Step 3 – Key findings
We calculate each result, then store it as a sentence in a list called `findings`. Each item is `[topic, sentence]`.

In [13]:
findings = []

# 1. Price level
prices = df["discount_price"].dropna()
text = ("Median selling price is Rs " + str(round(prices.median())) + " (average Rs " + str(round(prices.mean())) +
        "). Prices range from Rs " + str(round(prices.min())) + " to Rs " + str(round(prices.max())) + ".")
findings.append(["Price level", text])

# 2. Discounts
discounts = df["discount_percent"].dropna()
if len(discounts) > 0:
    text = "Verified discounts exist for " + str(len(discounts)) + " listings: average " + str(round(discounts.mean(), 1)) + "%."
else:
    text = "No listing has a verified MRP, so discounts could not be analysed."
findings.append(["Discounts", text])

# 3. Brands
brand_price = df.groupby("brand")["discount_price"].agg(["mean", "count"]).dropna()
brand_price = brand_price[brand_price["count"] >= 2]          # only brands with 2 or more priced listings
if len(brand_price) >= 2:
    priciest = brand_price["mean"].idxmax()
    cheapest = brand_price["mean"].idxmin()
    text = priciest + " is the priciest brand on average (Rs " + str(round(brand_price.loc[priciest, "mean"])) + \
           "); " + cheapest + " is the cheapest (Rs " + str(round(brand_price.loc[cheapest, "mean"])) + ")."
    findings.append(["Brand pricing", text])

brand_count = df["brand"].value_counts()
text = brand_count.index[0] + " has the most listings (" + str(brand_count.iloc[0]) + " of " + str(len(df)) + ")."
findings.append(["Brand presence", text])

# 4. Tonnage
common_ton = df["tonnage"].mode()[0]
share = (df["tonnage"] == common_ton).sum() / df["tonnage"].notna().sum() * 100
findings.append(["Tonnage", str(common_ton) + " ton is the most common capacity (" + str(round(share)) + "% of listings). "])

r = df["discount_price"].corr(df["tonnage"])
if r > 0.2:
    text = "Correlation r = " + str(round(r, 2)) + ": bigger capacity generally costs more."
else:
    text = "Correlation r = " + str(round(r, 2)) + ": capacity is not a clear price driver here."
findings.append(["Price vs tonnage", text])

# 5. Energy stars
common_star = int(df["energy_star"].mode()[0])
share = (df["energy_star"] == common_star).sum() / df["energy_star"].notna().sum() * 100
findings.append(["Energy rating", str(common_star) + "-star models are the most common (" + str(round(share)) + "% of listings). "])

price_5 = df.loc[df["energy_star"] == 5, "discount_price"].mean()
price_3 = df.loc[df["energy_star"] == 3, "discount_price"].mean()
findings.append(["Star premium", "5-star models average Rs " + str(round(price_5)) + " vs Rs " + str(round(price_3)) + " for 3-star."])

# 6. Price vs rating
r = df["discount_price"].corr(df["ratings"])
if r > 0.3:
    text = "Correlation r = " + str(round(r, 2)) + ": a higher price tends to go with better ratings."
else:
    text = "Correlation r = " + str(round(r, 2)) + ": a higher price does not guarantee a better customer rating."
findings.append(["Price vs rating", text])

# 7. Features
if "is_inverter" in df.columns:
    inverter_share = df["is_inverter"].mean() * 100
else:
    inverter_share = df["name"].str.contains("inverter", case=False, na=False).mean() * 100

if "has_wifi" in df.columns:
    wifi_share = df["has_wifi"].mean() * 100
    wifi_price = df.loc[df["has_wifi"] == True, "discount_price"].mean()
    no_wifi_price = df.loc[df["has_wifi"] == False, "discount_price"].mean()
else:
    has_wifi_temp = df["name"].str.contains("wifi|wi-fi", case=False, na=False)
    wifi_share = has_wifi_temp.mean() * 100
    wifi_price = df.loc[has_wifi_temp == True, "discount_price"].mean()
    no_wifi_price = df.loc[has_wifi_temp == False, "discount_price"].mean()

findings.append(["Technology", str(round(inverter_share)) + "% of listings are inverter ACs and " + str(round(wifi_share)) + "% mention Wi-Fi."])
findings.append(["Wi-Fi premium", "Wi-Fi models average Rs " + str(round(wifi_price)) + " vs Rs " + str(round(no_wifi_price)) + " without Wi-Fi."])

# 8. Best value
df["price_per_ton"] = df["discount_price"] / df["tonnage"]
best_value = df.dropna(subset=["price_per_ton"]).sort_values("price_per_ton").iloc[0]
findings.append(["Cheapest per ton", best_value["brand"] + " " + str(best_value["tonnage"]) + " ton gives the lowest price per ton: Rs " + str(round(best_value["price_per_ton"])) + "."])

for item in findings:
    print("-", item[0] + ":", item[1])

- Price level: Median selling price is Rs 39990 (average Rs 43103). Prices range from Rs 24990 to Rs 128800.
- Discounts: Verified discounts exist for 423 listings: average 29.1%.
- Brand pricing: Mitsubishi is the priciest brand on average (Rs 66200); Vestar is the cheapest (Rs 32481).
- Brand presence: Other has the most listings (103 of 652).
- Tonnage: 1.5 ton is the most common capacity (54% of listings). 
- Price vs tonnage: Correlation r = 0.67: bigger capacity generally costs more.
- Energy rating: 3-star models are the most common (60% of listings). 
- Star premium: 5-star models average Rs 46016 vs Rs 40769 for 3-star.
- Price vs rating: Correlation r = -0.05: a higher price does not guarantee a better customer rating.
- Technology: 68% of listings are inverter ACs and 8% mention Wi-Fi.
- Wi-Fi premium: Wi-Fi models average Rs 45172 vs Rs 42892 without Wi-Fi.
- Cheapest per ton: Mitsubishi 6.0 ton gives the lowest price per ton: Rs 11317.


In [14]:
pd.set_option("display.max_colwidth", 200)
findings_df = pd.DataFrame(findings, columns=["Topic", "Finding"])
findings_df

,Topic,Finding
0,Price level,Median selling price is Rs 39990 (average Rs 43103). Prices range from Rs 24990 to Rs 128800.
1,Discounts,Verified discounts exist for 423 listings: average 29.1%.
2,Brand pricing,Mitsubishi is the priciest brand on average (Rs 66200); Vestar is the cheapest (Rs 32481).
3,Brand presence,Other has the most listings (103 of 652).
4,Tonnage,1.5 ton is the most common capacity (54% of listings).
5,Price vs tonnage,Correlation r = 0.67: bigger capacity generally costs more.
6,Energy rating,3-star models are the most common (60% of listings).
7,Star premium,5-star models average Rs 46016 vs Rs 40769 for 3-star.
8,Price vs rating,Correlation r = -0.05: a higher price does not guarantee a better customer rating.
9,Technology,68% of listings are inverter ACs and 8% mention Wi-Fi.


## Step 4 – Advice
**For buyers**
- Compare *price per ton* instead of only the price, so different capacities can be compared fairly.
- A higher price does not always mean a better rating – check the number of ratings too.
- Higher energy-star models cost more upfront but save on electricity bills.

**For sellers**
- Look at which brands and price bands have many listings (more competition) and which have few.
- Features such as Wi-Fi and inverter technology can support a higher price.

**Limits of this study**
- It is a one-day snapshot, and prices change daily.
- Some brands have only 1–2 listings, so their averages are not reliable.
- Some MRP values were removed as invalid, so discount results use fewer listings.

## Step 5 – Save the findings

In [16]:
findings_df.to_csv( "Results_Key_Findings.csv", index=False)
print("Saved Results_Key_Findings.csv")

Saved Results_Key_Findings.csv


**End of the pipeline.** Raw → Filtered → Extracted → Cleaned → Aggregates / Analysis → Charts → Results.